In [ ]:
import streamlit as st
import ollama
import whisper
import os
from datetime import datetime
import pyttsx3

# 1. Initialize State and Models
if "messages" not in st.session_state:
    st.session_state.messages = []

@st.cache_resource
def load_whisper():
    return whisper.load_model("tiny")

stt_model = load_whisper()
engine = pyttsx3.init()

def speak_text(text):
    engine.say(text)
    engine.runAndWait()

# --- UI LAYOUT ---
st.set_page_config(page_title="Llama 3.2 Voice", layout="centered")
st.title("🎙️ Llama 3.2 Live Voice")

# Display chat history (The "Lively" Screen)
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# --- VOICE INPUT SECTION ---
audio_value = st.audio_input("Record your question")

if audio_value:
    # 2. Process Speech to Text
    with st.spinner("Listening..."):
        with open("temp_audio.wav", "wb") as f:
            f.write(audio_value.read())
        
        result = stt_model.transcribe("temp_audio.wav")
        user_text = result["text"]

    # 3. Add User Message to Screen
    st.session_state.messages.append({"role": "user", "content": user_text})
    with st.chat_message("user"):
        st.markdown(user_text)

    # 4. Generate & Stream Llama 3.2 Response
    with st.chat_message("assistant"):
        response_placeholder = st.empty()
        full_response = ""
        
        # Stream the text from Ollama
        stream = ollama.chat(
            model='llama3.2',
            messages=st.session_state.messages,
            stream=True,
        )

        for chunk in stream:
            full_response += chunk['message']['content']
            response_placeholder.markdown(full_response + "▌")
        
        response_placeholder.markdown(full_response)
        
    # 5. Speak the response
    st.session_state.messages.append({"role": "assistant", "content": full_response})
    speak_text(full_response)